# OFC 2026 공식 실측 데이터 검증 — 20 dB Unseen

공식 GitHub 저장소에서 **Train 530개 / TestGround 306개** 실제 EDFA 측정 조건을 읽어 물리 기반 Ridge 모델과 Gain/Tilt 기준선을 비교합니다.

**중요:** 별도의 물리 기반 Ridge 벤치마크이며 기존 PI-EDFA PyTorch 신경망의 검증 결과가 아닙니다. TestGround 정답을 훈련·모델 선택·스케일러 적합에 사용하지 않습니다.


In [ ]:
%pip -q install numpy pandas requests matplotlib

In [ ]:
from pathlib import Path
p = Path("/content/official_json_subset_benchmark.py")
p.write_text('#!/usr/bin/env python3\n"""Reproduce the independent 20-dB OFC 2026 measured-data sanity benchmark.\n\nIMPORTANT: This is a physics-informed spectral RIDGE SURROGATE, NOT the previous\nPI-EDFA PyTorch convolutional model. Data source is the official repository\'s\nmeasurement JSON (not Git-LFS CSV pointers). The test ground truth is used only\nfor one final score and not for training, normalization, or model selection.\n\nUsage (Python 3.10+):\n  pip install numpy pandas requests matplotlib\n  python official_json_subset_benchmark.py --output results_official_subset\n"""\nfrom __future__ import annotations\nimport argparse, json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport requests\nimport matplotlib.pyplot as plt\n\nORG_REPO = \'ofc-ml-challenge/ofc-ml-challenge-data-code\'\nBASE = f\'https://raw.githubusercontent.com/{ORG_REPO}/main/dataset/ML_challenge_admin\'\nDEVICES = (\'rdm1-lg1\', \'rdm2-lg1\', \'rdm3-co1\', \'rdm4-co1\', \'rdm5-co1\', \'rdm6-co1\')\nTRAIN = ([f\'aging/{part}\' for part in (\n    \'booster_rdm1-co1_fix_train.json\', \'booster_rdm2-co1_fix_train.json\',\n    \'preamp_rdm1-co1_fix_train.json\', \'preamp_rdm2-co1_fix_train.json\')]\n    + [f\'shb/preamp_{d}_random_train.json\' for d in (\'rdm1-co1\', \'rdm2-co1\')]\n    + [f\'unseen/tilt_booster_{d}_fix_train.json\' for d in DEVICES])\nTEST = [f\'unseen/gain_booster_{d}_fix_testGround.json\' for d in DEVICES]\n\n\ndef fetch(path: str, dataset: str, cachedir: Path) -> dict:\n    f = cachedir / dataset / path\n    if not f.exists():\n        f.parent.mkdir(parents=True, exist_ok=True)\n        url = f\'{BASE}/{dataset}/{path}\'\n        print(\'Downloading\', url, flush=True)\n        resp = requests.get(url, timeout=90)\n        resp.raise_for_status()\n        # The JSON files are ordinary Git blobs; do not confuse them with LFS pointers.\n        blob = resp.json()\n        f.write_text(json.dumps(blob), encoding=\'utf-8\')\n    return json.loads(f.read_text(encoding=\'utf-8\'))\n\n\ndef convert(path: str, dataset: str, cache: Path) -> list[dict]:\n    src = fetch(path, dataset, cache)\n    preamp = \'preamp\' in path\n    category = path.split(\'/\')[0]\n    import re\n    device = re.search(r\'rdm\\d+-(?:co|lg)\\d+\', path).group()\n    items = []\n    for m in src[\'measurement_data\']:\n        cfg = m[\'roadm_dut_preamp_info\'] if preamp else m[\'roadm_dut_edfa_info\']\n        inp = m[\'roadm_dut_preamp_input_power_spectra\'] if preamp else m[\'roadm_dut_wss_output_power_spectra\']\n        out = m[\'roadm_dut_wss_input_power_spectra\'] if preamp else m[\'roadm_dut_booster_output\']\n        loaded = (m[\'roadm_flatten_wss_active_channel_index\'] if preamp else\n                  m[\'roadm_dut_wss_active_channel_index\'])\n        # JSON object keys are the numbered channels 1..95.\n        p = np.array([float(inp[str(i)]) for i in range(1, 96)])\n        o = np.array([float(out[str(i)]) for i in range(1, 96)])\n        mask = np.zeros(95, dtype=bool)\n        mask[np.array(loaded, dtype=int)-1] = True\n        items.append({\'category\': category, \'device\': device,\n                      \'g\': float(cfg[\'target_gain\']),\n                      \'t\': float(cfg[\'target_gain_tilt\']),\n                      \'pin\': float(cfg[\'input_power\']),\n                      \'pout\': float(cfg[\'output_power\']),\n                      \'p\': p, \'y\': o-p, \'mask\': mask})\n    return items\n\n\ndef physical_features(s: dict) -> np.ndarray:\n    x = np.arange(95, dtype=float)/47. - 1.\n    mask, p = s[\'mask\'], s[\'p\']\n    count = max(1, int(mask.sum()))\n    avg = p[mask].mean() if mask.any() else 0.\n    powermw = np.where(mask, 10**(np.clip(p, -80, 20)/10), 0.)\n    share = powermw / max(1., powermw.sum()) if powermw.sum()==0 else powermw / powermw.sum()\n    dist = np.arange(95, dtype=float)[:, None] - np.arange(95, dtype=float)[None, :]\n    kernels = [1/(1+(dist/w)**2) @ share for w in (2,7,20)]\n    kernel_means = [v[mask].mean() if mask.any() else 0. for v in kernels]\n    dev = np.clip((p-avg)/10, -5, 5)\n    log_share = np.log1p(95*share)\n    occ = count/95\n    delta = s[\'pout\']-s[\'pin\']-s[\'g\']\n    pin = (s[\'pin\']+10)/20\n    gain = s[\'g\']-18\n    cols = [np.ones(95), x, x*x, x**3,\n            np.full(95, gain), gain*x,\n            np.full(95, s[\'t\']), s[\'t\']*x, s[\'t\']*x*x,\n            np.full(95, pin), pin*x, pin*x*x,\n            np.full(95, delta), delta*x, np.full(95, occ), occ*x,\n            dev, dev*x, log_share, log_share*x]\n    cols += [(v-mn)*25 for v,mn in zip(kernels,kernel_means)]\n    cols += [np.exp(-.5*((x-(-1+i/4))/.31)**2) for i in range(9)]\n    return np.column_stack(cols)\n\n\ndef baseline(s):\n    return s[\'g\']+.5*s[\'t\']*(np.arange(95)/47-1)\n\n\ndef ridge_fit(rows: list[dict], alpha: float = 2.):\n    F = [physical_features(s) for s in rows]\n    dim = F[0].shape[1]\n    X, y, weights = [], [], []\n    for s,f in zip(rows,F):\n        on = s[\'mask\']\n        X.append(f[on]);y.append((s[\'y\']-baseline(s))[on]);\n        weights.append(np.full(int(on.sum()), 1/max(1,int(on.sum()))))\n    X=np.concatenate(X); y=np.concatenate(y); w=np.concatenate(weights)\n    totw=w.sum();mu=(X*w[:,None]).sum(axis=0)/totw\n    std=np.maximum(.001,np.sqrt(np.maximum(0,(X*X*w[:,None]).sum(axis=0)/totw-mu**2)))\n    mu[0]=0;std[0]=1\n    Z=(X-mu)/std\n    gram=Z.T@(w[:,None]*Z)\n    regularizer=np.eye(dim)*alpha;regularizer[0,0]=1e-5\n    beta=np.linalg.solve(gram+regularizer, Z.T@(w*y))\n    return beta,mu,std\n\n\ndef prediction(s,coef):\n    beta,mu,std=coef\n    return baseline(s)+((physical_features(s)-mu)/std)@beta\n\n\ndef official_score(rows, outputs):\n    byrow=[]; flat=[]\n    for s,pred in zip(rows,outputs):\n        err=np.abs((s[\'y\']-pred)[s[\'mask\']])\n        if len(err):\n            byrow.append((err.mean(),err.std()))\n            flat.append(err)\n    flat=np.concatenate(flat)\n    mae=float(np.mean([x[0] for x in byrow]));std=float(np.mean([x[1] for x in byrow]))\n    p95=float(np.quantile(flat,.95));mx=float(flat.max())\n    score=mae+.15*std+.3*max(0,p95-mae-.5)+.1*max(0,mx-p95-.7)\n    return {\'score\':score,\'mae\':mae,\'std\':std,\'p95\':p95,\'max_error\':mx,\n            \'rmse\':float(np.sqrt(np.mean(flat**2))),\n            \'samples\':len(byrow),\'loaded_points\':len(flat),\n            \'within_0p2\':float(np.mean(flat<=.2))}\n\n\ndef run(output, cachedir):\n    train=[s for p in TRAIN for s in convert(p,\'Train\',cachedir)]\n    test=[s for p in TEST for s in convert(p,\'TestGround\',cachedir)]\n    assert len(train)==530 and len(test)==306, (len(train),len(test))\n    assert {s[\'g\'] for s in train}=={18} and {s[\'g\'] for s in test}=={20}\n    coef=ridge_fit(train)\n    b=[baseline(s) for s in test]\n    pred=[prediction(s,coef) for s in test]\n    outcome={\'setup\':{\'train_rows\':len(train),\'test_rows\':len(test),\n                      \'features\':32,\'ridge_alpha\':2,\n                      \'validation\':\'Only 20-dB gain / fixed-loading TestGround, post-contest data\',\n                      \'note\':\'This ridge surrogate is NOT the PI-EDFA neural model\'},\n             \'baseline\':official_score(test,b),\n             \'physics_ridge\':official_score(test,pred)}\n    devrecords=[]\n    for dev in DEVICES:\n        idx=[k for k,s in enumerate(test) if s[\'device\']==dev]\n        a=[test[k] for k in idx]\n        v={\'device\':dev,\'baseline\':official_score(a,[b[k] for k in idx]),\n           \'physics_ridge\':official_score(a,[pred[k] for k in idx])}\n        devrecords.append(v)\n    outcome[\'by_device\']=devrecords\n    output.mkdir(parents=True,exist_ok=True)\n    (output/\'metrics.json\').write_text(json.dumps(outcome,indent=2),encoding=\'utf-8\')\n    pd.DataFrame([{\'device\':v[\'device\'],\'baseline_score\':v[\'baseline\'][\'score\'],\n                   \'ridge_score\':v[\'physics_ridge\'][\'score\'],\n                   \'baseline_mae\':v[\'baseline\'][\'mae\'],\n                   \'ridge_mae\':v[\'physics_ridge\'][\'mae\']} for v in devrecords]).to_csv(\n                       output/\'device_scores.csv\',index=False)\n    fig,ax=plt.subplots(figsize=(8,4))\n    ax.bar([\'Gain/Tilt only\',\'Physics-guided ridge\'],[outcome[\'baseline\'][\'score\'],outcome[\'physics_ridge\'][\'score\']])\n    ax.set_ylabel(\'OFC 2026 score (lower is better)\')\n    ax.set_title(\'Official measured TestGround / Unseen 20 dB (6 devices)\')\n    fig.tight_layout();fig.savefig(output/\'score_comparison.png\',dpi=150);plt.close(fig)\n    print(json.dumps(outcome,indent=2))\n\n\nif __name__==\'__main__\':\n    ap=argparse.ArgumentParser();ap.add_argument(\'--output\',default=\'results_official_subset\')\n    ap.add_argument(\'--cache\',default=\'.ofc_raw_json_cache\');a=ap.parse_args()\n    run(Path(a.output),Path(a.cache))\n', encoding="utf-8")
print("Created",p)

In [ ]:
!python /content/official_json_subset_benchmark.py --output /content/ofc_20db_benchmark

In [ ]:
from IPython.display import display, Image
import pandas as pd, json
from pathlib import Path
out=Path('/content/ofc_20db_benchmark')
data=json.loads((out/'metrics.json').read_text())
print('Official score (baseline):',round(data['baseline']['score'],6))
print('Official score (physics ridge):',round(data['physics_ridge']['score'],6))
display(pd.read_csv(out/'device_scores.csv'))
display(Image(filename=str(out/'score_comparison.png')))